## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_groq import ChatGroq

MODEL = "llama-3.1-8b-instant"
DB_NAME = "vector_db"

load_dotenv(override=True)
llm = ChatGroq(
    model=MODEL,
    temperature=0
)

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6692.69it/s]


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [8]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [12]:
retriever.invoke("Who is Avery?")

[Document(id='2ac43b1f-6a8e-4a16-ad11-473177423f79', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [13]:
llm.invoke("Who is Avery?")

AIMessage(content='There are several notable individuals named Avery. Here are a few:\n\n1. **Avery Bradley**: An American professional basketball player who plays in the NBA. He has played for several teams, including the Boston Celtics, Detroit Pistons, and Los Angeles Clippers.\n2. **Avery Bradley (musician)**: An American musician and producer who has worked with artists such as Kanye West and Drake.\n3. **Avery Bradley (footballer)**: An American soccer player who has played for several teams, including the Seattle Sounders FC.\n4. **Avery Bradley (actor)**: An American actor who has appeared in films and television shows, including "The Walking Dead" and "The Vampire Diaries".\n5. **Avery Bradley (author)**: An American author who has written several books, including "The Avery Bradley Story" and "The Avery Bradley Chronicles".\n\nHowever, without more context, it\'s difficult to determine which Avery you are referring to. If you could provide more information, I\'d be happy to t

## Time to put this together!

In [11]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [16]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [18]:
answer_question("Who is Avery Lancaster?", [])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company to its current position as a leading Insurance Tech provider. Avery is known for her innovative leadership strategies and risk management expertise.'

## What could possibly come next? 

In [ ]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSA

## Admit it - you thought RAG would be more complicated than that!!